# Differentiable Marching Tetrahedra Pipeline
This notebook demonstrates how to load a mesh, sample points and normals, compute SDF values, and reconstruct the mesh using Delaunay triangulation and Marching Tetrahedra.

In [1]:
import torch
import trimesh
import conquer3d.data.assets.common as c3d_assets
import plotly.graph_objects as go
from conquer3d.data_structure import TriangleMesh
from conquer3d.ops import tetrahedralize, marching_tetrahedra

# 1. Load the Bunny mesh
bunny = c3d_assets.StanfordBunny()
vertices, faces, _ = bunny.get()

# Move tensors to GPU
vertices = vertices.cuda()
faces = faces.cuda()

# 2. Build the TriangleMesh C++ backend structure
mesh = TriangleMesh(vertices, faces.to(torch.int32))

# 3. Sample a point cloud (10000 points) uniformly from the surface, along with normals
# signature: sample_points(num_points, uniform, return_normals, return_colors, use_triangle_normal)
pts, tri_indices, normals, colors = mesh.sample_points(10000, True, True, False, False)

# 4. Create two new points for each point: p + s * n and p - s * n
s = 0.001
pts_out = pts + s * normals
pts_in = pts - s * normals

# Combine all 30000 points
all_pts = torch.cat([pts, pts_out, pts_in], dim=0)

# 5. Obtain the SDF values for all points using the BVH query_points method
# signature: query_points(query_pts, return_sdf, return_prj_pts, sign_mode, distance_mode)
query_ids, tri_ids, prj_pts, dists = mesh.query_points(all_pts, True, True, 0, 0)
sdfs = dists  # dists contains SDF values when return_sdf=True

# 6. Perform Delaunay Triangulation to create tetrahedra
tets = tetrahedralize(all_pts)

# 7. Perform Marching Tetrahedra to extract the isosurface
with torch.no_grad():
    mesh_v, mesh_f = marching_tetrahedra(all_pts, tets, sdfs)

# 8. Export and visualize the reconstructed mesh
out_mesh = trimesh.Trimesh(vertices=mesh_v.cpu().numpy(), faces=mesh_f.cpu().numpy())
out_mesh.export("reconstructed_bunny.obj")
print(f"Original mesh: {len(vertices)} vertices")
print(f"Reconstructed mesh: {len(mesh_v)} vertices, {len(mesh_f)} faces!")

# Extract vertices (x, y, z) and faces (i, j, k) for Plotly
x, y, z = mesh_v.cpu().numpy().T
i, j, k = mesh_f.cpu().numpy().T

fig = go.Figure(data=[
    go.Mesh3d(
        x=x, y=y, z=z,
        i=i, j=j, k=k,
        color='lightblue',
        opacity=0.8,
        flatshading=True
    )
])

fig.update_layout(
    scene=dict(
        xaxis=dict(showbackground=False, visible=False),
        yaxis=dict(showbackground=False, visible=False),
        zaxis=dict(showbackground=False, visible=False),
        aspectmode='data'
    ),
    margin=dict(l=0, r=0, b=0, t=0)
)
fig.show()


Reading stanford-bunny.obj...
Original mesh: 34834 vertices
Reconstructed mesh: 84055 vertices, 172907 faces!
